# Iceberg and ship classification

Two radar polarizations describe the same observation. This experiment learns image features
from those bands and tests whether incidence-angle metadata adds useful information.

The historical single-model public leaderboard scores were **0.19348** and **0.20507**.
The experiment below uses the original fixed local holdout; leaderboard scoring is a separate evaluation.

**Requirements:** the project environment and authorized Kaggle data. Set `ICEBERG_DATA_DIR`.
The default profile is the complete modern baseline. To inspect an existing completed experiment,
set `ICEBERG_EXISTING_RUN` to its run directory; its input and artifact fingerprints are verified.

## 1. Experiment configuration

The image-only baseline uses all training records outside the fixed holdout. The metadata profile
changes the input branch while keeping the split and training budget fixed. Choose the profile
before comparing results; a reduced budget answers a different experimental question.

In [ ]:
import csv
import json
import os
from datetime import UTC, datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display
from sklearn.model_selection import StratifiedShuffleSplit

from iceberg.data import load_data
from iceberg.runtime import evaluate, predict, train, verify_run

PROJECT = Path.cwd()
if not (PROJECT / "pyproject.toml").exists():
    PROJECT = PROJECT.parent
DATA_DIR = Path(os.environ.get("ICEBERG_DATA_DIR", PROJECT / ".local/data"))
PROFILE = os.environ.get("ICEBERG_PROFILE", "baseline")
CONFIG = json.loads((PROJECT / "configs" / f"{PROFILE}.json").read_text())
REUSE = os.environ.get("ICEBERG_EXISTING_RUN")
RUN_ROOT = Path(os.environ.get("ICEBERG_RUN_DIR", PROJECT / ".local/runs"))
RUN_DIR = Path(REUSE).resolve() if REUSE else RUN_ROOT / datetime.now(UTC).strftime("%Y%m%dT%H%M%S%f")
assert (DATA_DIR / "train.json").is_file(), "Download data and set ICEBERG_DATA_DIR first"
plt.rcParams.update({"figure.figsize": (8, 3.5), "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False})
print(json.dumps(CONFIG, indent=2))

## 2. Radar bands and incidence angle

The two 75×75 bands contain radar backscatter in dB, rather than ordinary RGB pixel values.
Keeping both channels aligned preserves their correspondence. A training-fitted channel
standardizer retains relative brightness differences between observations.

Missing incidence angles remain missing until preprocessing. The optional metadata branch
uses training-median imputation plus a missingness indicator, avoiding a fictitious physical angle.

In [ ]:
data = load_data(DATA_DIR / "train.json", labeled=True)
angles = data.angles.reshape(-1)
observed = np.isfinite(angles)
print({"records": len(data.ids), "radar_shape": data.images.shape,
       "iceberg_fraction": float(data.targets.mean()), "missing_angles": int((~observed).sum())})
fig, ax = plt.subplots()
ax.hist(angles[observed], bins=20, color="#176b75", edgecolor="white")
ax.set(title="Incidence-angle distribution in labeled observations",
       xlabel="Incidence angle (degrees; missing values excluded)", ylabel="Records")
fig.tight_layout()
plt.show()

The histogram describes coverage, not predictive value. Incidence angle can encode acquisition
structure, so a random stratified split alone does not establish performance on new acquisitions.
An angle-grouped evaluation is a separate robustness check; angle lookup against labels is not part of this model.

## 3. Validation split

Retain the original seed-25 stratified holdout of 256 records. Fit every preprocessing statistic
on the remaining training rows. The holdout is used for checkpoint selection, so it is not a
second untouched test set.

In [ ]:
split = StratifiedShuffleSplit(n_splits=1, test_size=CONFIG["validation_fraction"],
                              random_state=CONFIG["split_seed"])
train_indices, valid_indices = next(split.split(data.images, data.targets))
assert not set(train_indices) & set(valid_indices)
print({"training_rows": len(train_indices), "holdout_rows": len(valid_indices),
       "training_iceberg_fraction": float(data.targets[train_indices].mean()),
       "holdout_iceberg_fraction": float(data.targets[valid_indices].mean())})

## 4. Convolutional model

```text
Two radar bands → training-fitted scaling → four CNN stages → average + max pooling ─┐
                                                                                   ├→ dense head → probability
Incidence angle → training-fitted imputation/scaling + missing flag → optional branch ┘
```

Progressive downsampling reduces the cost of repeated full-resolution convolutions. Native
BatchNorm2d normalizes convolutional activations; LayerNorm avoids dependence on the last
minibatch in the dense head. Standard AdamW and a cosine schedule train over ordinary epochs.

Spatial flips, right-angle rotations and mild affine transforms are shared across the two bands.
Extreme zoom and RGB color augmentation would change the physical input assumptions and are omitted.

## 5. Training and checkpoint selection

The best checkpoint minimizes held-out log loss. The run directory retains the resolved recipe,
preprocessing parameters, actual split, training history, source/input identities and checksums.
An existing run is reused only after verification.

In [ ]:
if REUSE:
    manifest = verify_run(RUN_DIR, DATA_DIR / "train.json")
    assert manifest["config"] == CONFIG, "Select the profile that produced this run"
else:
    manifest = train(DATA_DIR / "train.json", RUN_DIR, CONFIG)
metrics = json.loads((RUN_DIR / "metrics.json").read_text())
display(Markdown(
    f"**Held-out log loss: {metrics['validation_log_loss']:.4f}.** "
    f"Training used {manifest['train_rows']} records; evaluation used "
    f"{manifest['validation_rows']} holdout records. This is a local validation metric."
))
history = json.loads((RUN_DIR / "history.json").read_text())
fig, ax = plt.subplots()
ax.plot([r["epoch"] for r in history], [r["validation_log_loss"] for r in history],
        color="#176b75", marker="o", markersize=3)
ax.set(title="Checkpoint selection uses held-out log loss", xlabel="Epoch", ylabel="Validation log loss")
fig.tight_layout()
plt.show()

## 6. Holdout predictions

Reload the saved checkpoint and preprocessing before inspecting its errors. Distributional
separation is useful context but does not replace log loss or establish probability calibration.
Use the same inference and test-time augmentation settings for validation and submission.

In [ ]:
reloaded = evaluate(RUN_DIR / "model.pt", DATA_DIR / "train.json")
assert abs(reloaded["validation_log_loss"] - metrics["validation_log_loss"]) < 1e-5
with (RUN_DIR / "validation.csv").open() as stream:
    rows = list(csv.DictReader(stream))
fig, ax = plt.subplots()
for target, label, color in [(0, "Ship", "#263f59"), (1, "Iceberg", "#168376")]:
    values = [float(r["probability"]) for r in rows if int(r["target"]) == target]
    ax.hist(values, bins=np.linspace(0, 1, 16), alpha=0.6, label=label, color=color)
ax.set(title="Predictions on the fixed holdout", xlabel="Predicted iceberg probability",
       ylabel="Records")
ax.legend()
fig.tight_layout()
plt.show()

The overlap identifies uncertainty and confident mistakes. Compare image-only and metadata
experiments with the same split, seed and inference recipe. Improvements on this small holdout
need confirmation on the competition evaluation and an acquisition-grouped split.

## 7. Test predictions

Set `ICEBERG_FULL_TEST=1` for all 8,424 authorized test records. Prediction retains their IDs and
order, and writes a receipt binding the output to its input and checkpoint. Producing a CSV and
submitting it for a leaderboard score are distinct operations.

In [ ]:
if os.environ.get("ICEBERG_FULL_TEST", "0") == "1":
    receipt = predict(RUN_DIR / "model.pt", DATA_DIR / "test.json", RUN_DIR / "submission.csv")
    print(receipt)
else:
    print("Full-test inference disabled; set ICEBERG_FULL_TEST=1 to enable it.")

## 8. Experiment artifacts

Local artifacts remain available without a tracking server. The optional MLflow import checks
completed-run fingerprints and logs them to an existing deployment. Docker Lab can expose an
existing SeaweedFS client mount read-only. Service deployment and storage administration remain external.

The Kaggle entry notebook runs the package from an attached code wheel and saves a portable
run bundle without requiring access to private tracking infrastructure.

[Results](../docs/results.md) · [Method changes](../docs/migration.md) ·
[Runbook](../docs/reproduction.md) · [Tracking](../docs/mlops.md)